## Before running: paths and an existing timing result

Start Jupyter with the README's `RF_MOUSE_DIR`, `RF_DATE`, and `RF_SESSION`
environment variables set. The parameter cell reads these values. Check the
acquisition channel separately. Linux/macOS users can use the current code
with their mounted paths. Native Windows users can set
`$env:RF_MOUSE_DIR = "D:\recordings\mouse_01"`; follow the
[PowerShell setup](docs/platforms_and_data_safety.md) before starting Jupyter.
A remote Linux kernel uses Linux settings regardless of the browser's OS.

**This is preview code. Reviewing the result after it finishes is a critical
step before saving or RF generation. A completed run or matching count alone
does not establish correct timing. Apply this rule to every preview and
diagnostic plot in this workflow.**

Keep `is_save_on_list_time = False` for the first run. Inspect the
trial count, first/last boundaries, and every repaired gap before enabling
saving. With `True`, the final cell **replaces an existing**
`data/on_list_times.npy` without asking. Preserve an earlier verified
file before saving a replacement; the manual notebook writes this same path.

This auto route does **not** read or update `session_slices.sqlite3`.
Keep any existing correction database intact, and do not delete it to switch
routes or apply its old delete indices to this already-cleaned sequence.
If you later use `matlab.ipynb`, read its database/Windows note first:
the manual route uses Linux/macOS-specific SQLite locking until the documented
native Windows connection change is made.

[Conversion workflow](README.md) ·
[Platform setup, backups, and other cache assumptions](docs/platforms_and_data_safety.md)


In [ ]:
import os
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

from Utils.load_files import load_binary
from Utils.recording import interp_replace
from Utils.Sessions import Session
from Utils.ttl_utils import fill_short_gaps


In [ ]:
"""
Change all the variables below to match recording.
"""

base_dir = os.environ["RF_MOUSE_DIR"]
date: str | int = os.environ["RF_DATE"]
num_of_rec: int = int(os.environ["RF_SESSION"])

# This is a 0 indexed channel number.
analogue_input_channel: int = 3


is_convert_to_zero: bool = False
is_save_on_list_time: bool = False
# Stored session edits are not applied to this auto-cleaned list.
# Applying the old delete indices again would delete valid edges.

edge_threshold: int = 1000
maximum_internal_low_gap_samples: int = 300
maximum_short_high_run_samples: int = 300

periodic_support_interval_count: int = 20
typical_interval_tolerance_fraction: float = 0.20
gap_repair_phase_tolerance_fraction: float = 0.10
final_boundary_interval_seconds: float = 0.1

"""------------------------------------------------------------------"""

subfolder_filler = f"{date}_{num_of_rec}"
base_dir = f"{base_dir}/{date}/{subfolder_filler}"

oebin_file_dir = next(
    (Path(base_dir) / date).glob(
        "*/experiment1/recording1/structure.oebin"
    )
)

session = Session(oebin_file_dir)
session_info = session.get_session_info()

base_data_dir: str = session_info["base_path"]
record_nodes: str = session_info["record_nodes"]
recording_name: str = session_info["recording_name"]
experiment_id: str = session_info["experiment_id"]

path_between = f"/{record_nodes}/{experiment_id}/"
continuous_folder = (
    base_data_dir
    + path_between
    + recording_name
    + "/continuous/"
)

print("==============================")
print("Loading ADC data...")

ADC_name: str = session_info["continuous_ADC_folder"]
analogue_total_input_channel_number: int = session_info[
    "ADC_input_channel"
]
ADC_sampling_rate: float = session_info["ADC_sample_rate"]
ADC_datafile = continuous_folder + ADC_name + "/continuous.dat"

photodiode_data = load_binary(
    ADC_datafile,
    analogue_total_input_channel_number,
    analogue_input_channel,
)

ADC_continuous_time_stamp_file = (
    f"{continuous_folder}/{ADC_name}/timestamps.npy"
)
ADC_continuous_time_stamp_data_raw = np.load(
    ADC_continuous_time_stamp_file,
    mmap_mode="r",
)

if is_convert_to_zero:
    ADC_continuous_time_stamp_data = (
        ADC_continuous_time_stamp_data_raw
        - ADC_continuous_time_stamp_data_raw[0]
    )
else:
    ADC_continuous_time_stamp_data = (
        ADC_continuous_time_stamp_data_raw
    )

trials = loadmat(f"{base_dir}/{date}.mat")["trials"]

assert len(ADC_continuous_time_stamp_data) == len(photodiode_data), (
    "Length mismatch: "
    f"{len(ADC_continuous_time_stamp_data)} vs "
    f"{len(photodiode_data)}"
)

ADC_duration_seconds = len(photodiode_data) / ADC_sampling_rate
print(f"ADC samples: {len(photodiode_data)}")
print(f"ADC sampling rate: {ADC_sampling_rate} Hz")
print(f"ADC duration: {ADC_duration_seconds} seconds")
print(f"Trial count: {len(trials)}")


In [ ]:
# Convert the raw ADC signal to a digital state without filtering it.
photodiode_is_high = photodiode_data > edge_threshold
raw_transition_count = int(
    np.count_nonzero(
        photodiode_is_high[1:] != photodiode_is_high[:-1]
    )
)

# First close short LOW cracks inside a HIGH period.
photodiode_is_high_after_low_gap_fill = fill_short_gaps(
    photodiode_is_high,
    min_gap=maximum_internal_low_gap_samples,
)

# Then remove short HIGH islands inside a LOW period.
photodiode_is_high_clean = ~fill_short_gaps(
    ~photodiode_is_high_after_low_gap_fill,
    min_gap=maximum_short_high_run_samples,
)

filled_digital_state_changes = np.diff(
    photodiode_is_high_after_low_gap_fill.astype(np.int8)
)
filled_transition_sample_indices = (
    np.flatnonzero(filled_digital_state_changes != 0) + 1
)

print("==============================")
print(f"Edge threshold: {edge_threshold}")
print(
    "Maximum internal LOW gap: "
    f"{maximum_internal_low_gap_samples / ADC_sampling_rate * 1000:.3f} ms"
)
print(
    "Maximum short HIGH run: "
    f"{maximum_short_high_run_samples / ADC_sampling_rate * 1000:.3f} ms"
)
print(f"Raw threshold transitions: {raw_transition_count}")
print(
    "After filling short LOW gaps: "
    f"{len(filled_transition_sample_indices)}"
)


In [ ]:
# Do not invent an edge at either recording boundary.
if photodiode_is_high_clean[0]:
    print(
        "Recording starts HIGH; the preceding rising edge was "
        "not recorded and was not synthesized."
    )
if photodiode_is_high_clean[-1]:
    print(
        "Recording ends HIGH; the following falling edge was "
        "not recorded and was not synthesized."
    )

clean_digital_state_changes = np.diff(
    photodiode_is_high_clean.astype(np.int8)
)

rising_edge_sample_indices = (
    np.flatnonzero(clean_digital_state_changes == 1) + 1
)
falling_edge_sample_indices = (
    np.flatnonzero(clean_digital_state_changes == -1) + 1
)

transition_sample_indices = np.sort(
    np.concatenate(
        (rising_edge_sample_indices, falling_edge_sample_indices)
    )
)
transition_edge_directions = clean_digital_state_changes[
    transition_sample_indices - 1
]
transition_times = ADC_continuous_time_stamp_data[
    transition_sample_indices
]

assert abs(
    len(rising_edge_sample_indices)
    - len(falling_edge_sample_indices)
) <= 1
assert np.all(np.diff(transition_sample_indices) > 0)
assert abs(len(transition_sample_indices) - len(trials)) <= 1000, (
    "Detected transition count differs from the trial count by more "
    "than 1000. Check the ADC channel and threshold."
)

# Every retained edge must be an exact raw ADC threshold crossing.
assert np.all(
    photodiode_data[rising_edge_sample_indices - 1]
    <= edge_threshold
)
assert np.all(
    photodiode_data[rising_edge_sample_indices]
    > edge_threshold
)
assert np.all(
    photodiode_data[falling_edge_sample_indices - 1]
    > edge_threshold
)
assert np.all(
    photodiode_data[falling_edge_sample_indices]
    <= edge_threshold
)

clean_transition_positions_in_filled_edges = np.searchsorted(
    filled_transition_sample_indices,
    transition_sample_indices,
)
assert np.array_equal(
    filled_transition_sample_indices[
        clean_transition_positions_in_filled_edges
    ],
    transition_sample_indices,
)

filled_edge_survived_high_run_cleanup = np.zeros(
    len(filled_transition_sample_indices),
    dtype=bool,
)
filled_edge_survived_high_run_cleanup[
    clean_transition_positions_in_filled_edges
] = True
removed_short_high_edge_indices = np.flatnonzero(
    ~filled_edge_survived_high_run_cleanup
)
removed_short_high_edge_sample_indices = (
    filled_transition_sample_indices[removed_short_high_edge_indices]
)

assert len(removed_short_high_edge_sample_indices) % 2 == 0

print("==============================")
print(
    "After removing short HIGH runs: "
    f"{len(transition_sample_indices)}"
)
print(f"Rising edges: {len(rising_edge_sample_indices)}")
print(f"Falling edges: {len(falling_edge_sample_indices)}")

for removed_edge_pair_start in range(
    0,
    len(removed_short_high_edge_sample_indices),
    2,
):
    removed_high_run_start_sample = int(
        removed_short_high_edge_sample_indices[
            removed_edge_pair_start
        ]
    )
    removed_high_run_end_sample = int(
        removed_short_high_edge_sample_indices[
            removed_edge_pair_start + 1
        ]
    )
    removed_high_run_duration_ms = (
        (
            removed_high_run_end_sample
            - removed_high_run_start_sample
        )
        / ADC_sampling_rate
        * 1000
    )
    print(
        "Removed short HIGH run: samples "
        f"[{removed_high_run_start_sample}, "
        f"{removed_high_run_end_sample}), "
        f"{removed_high_run_duration_ms:.6f} ms"
    )


In [ ]:
transition_intervals_seconds = np.diff(transition_times)
typical_edge_interval_seconds = float(
    np.median(transition_intervals_seconds)
)

typical_interval_mask = (
    np.abs(
        transition_intervals_seconds
        - typical_edge_interval_seconds
    )
    <= typical_interval_tolerance_fraction
    * typical_edge_interval_seconds
)

typical_interval_cumulative_count = np.concatenate(
    (
        [0],
        np.cumsum(typical_interval_mask, dtype=np.int64),
    )
)
typical_interval_window_counts = (
    typical_interval_cumulative_count[
        periodic_support_interval_count:
    ]
    - typical_interval_cumulative_count[
        :-periodic_support_interval_count
    ]
)
periodic_window_start_indices = np.flatnonzero(
    typical_interval_window_counts
    == periodic_support_interval_count
)

assert periodic_window_start_indices.size > 0, (
    "No stable periodic transition train was detected."
)

trial_block_start_edge_index = int(
    periodic_window_start_indices[0]
)
trial_block_stop_edge_index = int(
    periodic_window_start_indices[-1]
    + periodic_support_interval_count
    + 1
)

trim_start_count = trial_block_start_edge_index
trim_end_count = (
    len(transition_times) - trial_block_stop_edge_index
)

matched_transition_sample_indices = transition_sample_indices[
    trial_block_start_edge_index:trial_block_stop_edge_index
]
matched_transition_edge_directions = transition_edge_directions[
    trial_block_start_edge_index:trial_block_stop_edge_index
]
matched_transition_times = transition_times[
    trial_block_start_edge_index:trial_block_stop_edge_index
]

post_trim_start_boundary_intervals_seconds = np.diff(
    matched_transition_times[
        :periodic_support_interval_count + 1
    ]
)
post_trim_end_boundary_intervals_seconds = np.diff(
    matched_transition_times[
        -periodic_support_interval_count - 1:
    ]
)
boundary_interval_deviation_limit_seconds = (
    typical_interval_tolerance_fraction
    * typical_edge_interval_seconds
)
assert np.all(
    np.abs(
        post_trim_start_boundary_intervals_seconds
        - typical_edge_interval_seconds
    )
    <= boundary_interval_deviation_limit_seconds
)
assert np.all(
    np.abs(
        post_trim_end_boundary_intervals_seconds
        - typical_edge_interval_seconds
    )
    <= boundary_interval_deviation_limit_seconds
)

print("==============================")
print(
    f"Typical edge interval: "
    f"{typical_edge_interval_seconds:.9f} seconds"
)
print(f"Trimmed transitions from start: {trim_start_count}")
print(f"Trimmed transitions from end: {trim_end_count}")
print(
    "Transitions in detected trial block: "
    f"{len(matched_transition_times)}"
)
print(
    "Post-trim start interval range: "
    f"{post_trim_start_boundary_intervals_seconds.min() * 1000:.6f} "
    "to "
    f"{post_trim_start_boundary_intervals_seconds.max() * 1000:.6f} ms"
)
print(
    "Post-trim end interval range: "
    f"{post_trim_end_boundary_intervals_seconds.min() * 1000:.6f} "
    "to "
    f"{post_trim_end_boundary_intervals_seconds.max() * 1000:.6f} ms"
)


In [ ]:
matched_transition_intervals_seconds = np.diff(
    matched_transition_times
)
abnormal_interval_mask = (
    np.abs(
        matched_transition_intervals_seconds
        - typical_edge_interval_seconds
    )
    > typical_interval_tolerance_fraction
    * typical_edge_interval_seconds
)
abnormal_interval_indices = np.flatnonzero(
    abnormal_interval_mask
)

abnormal_interval_groups = []
if abnormal_interval_indices.size > 0:
    abnormal_group_split_positions = (
        np.flatnonzero(np.diff(abnormal_interval_indices) > 1)
        + 1
    )
    abnormal_interval_groups = np.split(
        abnormal_interval_indices,
        abnormal_group_split_positions,
    )

gap_analysis = []
gap_repair_candidate_indices = []
gap_repair_candidate_count_changes = []

for abnormal_interval_group in abnormal_interval_groups:
    left_anchor_edge_index = int(abnormal_interval_group[0])
    right_anchor_edge_index = int(
        abnormal_interval_group[-1] + 1
    )
    observed_interval_count = (
        right_anchor_edge_index - left_anchor_edge_index
    )
    anchor_span_seconds = float(
        matched_transition_times[right_anchor_edge_index]
        - matched_transition_times[left_anchor_edge_index]
    )
    expected_interval_count = int(
        round(anchor_span_seconds / typical_edge_interval_seconds)
    )
    transition_count_change = (
        expected_interval_count - observed_interval_count
    )
    phase_error = abs(
        anchor_span_seconds / typical_edge_interval_seconds
        - expected_interval_count
    )
    abnormal_group_intervals_seconds = (
        matched_transition_intervals_seconds[
            abnormal_interval_group
        ]
    )
    contains_short_interval = bool(
        np.any(
            abnormal_group_intervals_seconds
            < (
                1 - typical_interval_tolerance_fraction
            )
            * typical_edge_interval_seconds
        )
    )
    contains_long_interval = bool(
        np.any(
            abnormal_group_intervals_seconds
            > (
                1 + typical_interval_tolerance_fraction
            )
            * typical_edge_interval_seconds
        )
    )
    left_anchor_edge_direction = int(
        matched_transition_edge_directions[
            left_anchor_edge_index
        ]
    )
    right_anchor_edge_direction = int(
        matched_transition_edge_directions[
            right_anchor_edge_index
        ]
    )
    expected_right_anchor_edge_direction = (
        left_anchor_edge_direction
        if expected_interval_count % 2 == 0
        else -left_anchor_edge_direction
    )
    anchor_edge_polarity_matches = (
        right_anchor_edge_direction
        == expected_right_anchor_edge_direction
    )
    is_gap_repair_candidate = (
        transition_count_change > 0
        and phase_error <= gap_repair_phase_tolerance_fraction
        and contains_short_interval
        and contains_long_interval
        and anchor_edge_polarity_matches
    )

    gap_analysis.append(
        {
            "left_anchor_edge_index": left_anchor_edge_index,
            "right_anchor_edge_index": right_anchor_edge_index,
            "anchor_span_seconds": anchor_span_seconds,
            "observed_interval_count": observed_interval_count,
            "expected_interval_count": expected_interval_count,
            "transition_count_change": transition_count_change,
            "phase_error": phase_error,
            "contains_short_interval": contains_short_interval,
            "contains_long_interval": contains_long_interval,
            "anchor_edge_polarity_matches": (
                anchor_edge_polarity_matches
            ),
            "is_gap_repair_candidate": is_gap_repair_candidate,
        }
    )

    if is_gap_repair_candidate:
        gap_repair_candidate_indices.append(
            len(gap_analysis) - 1
        )
        gap_repair_candidate_count_changes.append(
            transition_count_change
        )

required_transition_count_change = (
    len(trials) - len(matched_transition_times)
)

print("==============================")
print(
    "Required transition count change: "
    f"{required_transition_count_change:+d}"
)

for gap_analysis_index, gap_information in enumerate(gap_analysis):
    print(
        f"Gap group {gap_analysis_index}: edges "
        f"{gap_information['left_anchor_edge_index']} to "
        f"{gap_information['right_anchor_edge_index']}, "
        f"span={gap_information['anchor_span_seconds']:.9f} s, "
        f"observed_intervals="
        f"{gap_information['observed_interval_count']}, "
        f"expected_intervals="
        f"{gap_information['expected_interval_count']}, "
        f"count_change="
        f"{gap_information['transition_count_change']:+d}, "
        f"phase_error="
        f"{gap_information['phase_error']:.6f}, "
        f"short+long="
        f"{gap_information['contains_short_interval'] and gap_information['contains_long_interval']}, "
        f"polarity_match="
        f"{gap_information['anchor_edge_polarity_matches']}, "
        f"repair_candidate="
        f"{gap_information['is_gap_repair_candidate']}"
    )

assert required_transition_count_change >= 0, (
    "The detected trial block still has extra transitions. "
    "Inspect the printed gap groups instead of trimming them silently."
)

selected_gap_repair_indices = []
if required_transition_count_change > 0:
    matching_gap_repair_plans = []

    for repair_group_count in range(
        1,
        len(gap_repair_candidate_indices) + 1,
    ):
        for candidate_positions in combinations(
            range(len(gap_repair_candidate_indices)),
            repair_group_count,
        ):
            proposed_count_change = sum(
                gap_repair_candidate_count_changes[
                    candidate_position
                ]
                for candidate_position in candidate_positions
            )
            if (
                proposed_count_change
                == required_transition_count_change
            ):
                matching_gap_repair_plans.append(
                    candidate_positions
                )

    print(
        "Matching automatic gap repair plans: "
        f"{len(matching_gap_repair_plans)}"
    )
    assert len(matching_gap_repair_plans) == 1, (
        "Gap repair is ambiguous. Review the printed candidates; "
        "no interpolation was applied."
    )

    selected_candidate_positions = matching_gap_repair_plans[0]
    selected_gap_repair_indices = [
        gap_repair_candidate_indices[candidate_position]
        for candidate_position in selected_candidate_positions
    ]

gap_repair_net_transition_count_change = sum(
    gap_analysis[gap_analysis_index][
        "transition_count_change"
    ]
    for gap_analysis_index in selected_gap_repair_indices
)
applied_gap_repairs = []
for gap_analysis_index in selected_gap_repair_indices:
    gap_information = gap_analysis[gap_analysis_index]
    left_anchor_edge_index = gap_information[
        "left_anchor_edge_index"
    ]
    right_anchor_edge_index = gap_information[
        "right_anchor_edge_index"
    ]
    applied_gap_repairs.append(
        {
            "left_anchor_edge_index_in_trial_block": (
                left_anchor_edge_index
            ),
            "right_anchor_edge_index_in_trial_block": (
                right_anchor_edge_index
            ),
            "left_anchor_sample_index": int(
                matched_transition_sample_indices[
                    left_anchor_edge_index
                ]
            ),
            "right_anchor_sample_index": int(
                matched_transition_sample_indices[
                    right_anchor_edge_index
                ]
            ),
            "observed_interval_count": gap_information[
                "observed_interval_count"
            ],
            "expected_interval_count": gap_information[
                "expected_interval_count"
            ],
            "net_transition_count_change": gap_information[
                "transition_count_change"
            ],
            "phase_error_fraction": gap_information[
                "phase_error"
            ],
        }
    )

on_list_time = matched_transition_times.copy()
on_list_edge_directions = (
    matched_transition_edge_directions.copy()
)
matched_observed_edge_is_retained = np.ones(
    len(matched_transition_times),
    dtype=bool,
)
interpolated_edge_time_segments = []

selected_gap_repair_indices_descending = sorted(
    selected_gap_repair_indices,
    key=lambda gap_index: gap_analysis[gap_index][
        "left_anchor_edge_index"
    ],
    reverse=True,
)

for gap_analysis_index in selected_gap_repair_indices_descending:
    gap_information = gap_analysis[gap_analysis_index]
    left_anchor_edge_index = gap_information[
        "left_anchor_edge_index"
    ]
    right_anchor_edge_index = gap_information[
        "right_anchor_edge_index"
    ]
    expected_interval_count = gap_information[
        "expected_interval_count"
    ]

    replacement_edge_count = expected_interval_count + 1
    replacement_edge_times = np.linspace(
        on_list_time[left_anchor_edge_index],
        on_list_time[right_anchor_edge_index],
        replacement_edge_count,
    )
    replacement_edge_directions = (
        matched_transition_edge_directions[
            left_anchor_edge_index
        ]
        * np.where(
            np.arange(replacement_edge_count) % 2 == 0,
            1,
            -1,
        )
    )

    matched_observed_edge_is_retained[
        left_anchor_edge_index + 1:right_anchor_edge_index
    ] = False
    interpolated_edge_time_segments.append(
        replacement_edge_times[1:-1]
    )

    on_list_time = interp_replace(
        on_list_time,
        left_anchor_edge_index,
        right_anchor_edge_index,
        new_length=replacement_edge_count,
    )
    on_list_edge_directions = np.concatenate(
        (
            on_list_edge_directions[:left_anchor_edge_index],
            replacement_edge_directions,
            on_list_edge_directions[
                right_anchor_edge_index + 1:
            ],
        )
    )

if interpolated_edge_time_segments:
    interpolated_edge_times = np.sort(
        np.concatenate(interpolated_edge_time_segments)
    )
else:
    interpolated_edge_times = np.array([], dtype=float)

final_observed_transition_sample_indices = (
    matched_transition_sample_indices[
        matched_observed_edge_is_retained
    ]
)
final_observed_positions_in_filled_edges = np.searchsorted(
    filled_transition_sample_indices,
    final_observed_transition_sample_indices,
)
assert np.array_equal(
    filled_transition_sample_indices[
        final_observed_positions_in_filled_edges
    ],
    final_observed_transition_sample_indices,
)

filled_edge_is_retained_in_final_on_list = np.zeros(
    len(filled_transition_sample_indices),
    dtype=bool,
)
filled_edge_is_retained_in_final_on_list[
    final_observed_positions_in_filled_edges
] = True
automatically_deleted_edge_indices = np.flatnonzero(
    ~filled_edge_is_retained_in_final_on_list
)
automatically_deleted_edge_sample_indices = (
    filled_transition_sample_indices[
        automatically_deleted_edge_indices
    ]
)
automatically_deleted_edge_directions = (
    filled_digital_state_changes[
        automatically_deleted_edge_sample_indices - 1
    ]
)
automatically_deleted_edge_times = (
    ADC_continuous_time_stamp_data[
        automatically_deleted_edge_sample_indices
    ]
)

trimmed_boundary_edge_sample_indices = np.concatenate(
    (
        transition_sample_indices[:trial_block_start_edge_index],
        transition_sample_indices[trial_block_stop_edge_index:],
    )
)
trimmed_boundary_edge_indices = np.searchsorted(
    filled_transition_sample_indices,
    trimmed_boundary_edge_sample_indices,
)
gap_replaced_observed_edge_sample_indices = (
    matched_transition_sample_indices[
        ~matched_observed_edge_is_retained
    ]
)
gap_replaced_observed_edge_indices = np.searchsorted(
    filled_transition_sample_indices,
    gap_replaced_observed_edge_sample_indices,
)

for gap_analysis_index, gap_information in enumerate(gap_analysis):
    if gap_analysis_index in selected_gap_repair_indices:
        gap_decision = (
            "INTERPOLATE: replace "
            f"{gap_information['observed_interval_count']} "
            "observed intervals with "
            f"{gap_information['expected_interval_count']}, "
            "net count change "
            f"{gap_information['transition_count_change']:+d}"
        )
    elif required_transition_count_change == 0:
        gap_decision = "PRESERVE: trial count already matches"
    elif gap_information["is_gap_repair_candidate"]:
        gap_decision = "PRESERVE: not selected by global count"
    else:
        gap_decision = "PRESERVE: phase does not match"

    print(f"Gap group {gap_analysis_index}: {gap_decision}")

automatic_edit_summary = {
    "detector": {
        "edge_threshold": edge_threshold,
        "maximum_internal_low_gap_samples": (
            maximum_internal_low_gap_samples
        ),
        "maximum_short_high_run_samples": (
            maximum_short_high_run_samples
        ),
    },
    "trim_start_count": trim_start_count,
    "trim_end_count": trim_end_count,
    "removed_short_high_edge_indices_in_low_gap_filled_edges": (
        removed_short_high_edge_indices.tolist()
    ),
    "trimmed_boundary_edge_indices_in_low_gap_filled_edges": (
        trimmed_boundary_edge_indices.tolist()
    ),
    "gap_replaced_observed_edge_indices_in_low_gap_filled_edges": (
        gap_replaced_observed_edge_indices.tolist()
    ),
    "deleted_edge_indices_in_low_gap_filled_edges": (
        automatically_deleted_edge_indices.tolist()
    ),
    "deleted_edge_sample_indices": (
        automatically_deleted_edge_sample_indices.tolist()
    ),
    "deleted_edge_directions": (
        automatically_deleted_edge_directions.tolist()
    ),
    "deleted_edge_times_seconds": (
        automatically_deleted_edge_times.tolist()
    ),
    "gap_repairs": applied_gap_repairs,
    "gap_repair_net_count_change": (
        gap_repair_net_transition_count_change
    ),
    "synthetic_replacement_edge_count": (
        len(interpolated_edge_times)
    ),
    "synthetic_replacement_edge_times_seconds": (
        interpolated_edge_times.tolist()
    ),
    "final_rising_edge_count": int(
        np.count_nonzero(on_list_edge_directions == 1)
    ),
    "final_falling_edge_count": int(
        np.count_nonzero(on_list_edge_directions == -1)
    ),
}

assert len(on_list_time) == len(trials), (
    f"Length mismatch: {len(on_list_time)} vs {len(trials)}"
)
assert len(on_list_edge_directions) == len(on_list_time)
assert np.all(
    on_list_edge_directions[1:]
    == -on_list_edge_directions[:-1]
)
assert np.all(np.diff(on_list_time) > 0)

print("==============================")
print(f"After LOW-gap fill: {len(filled_transition_sample_indices)}")
print(f"After HIGH-glitch removal: {len(transition_times)}")
print(f"After automatic trim: {len(matched_transition_times)}")
print(
    "Gap repair net count change: "
    f"{gap_repair_net_transition_count_change:+d}"
)
print(
    "Synthetic replacement edges: "
    f"{len(interpolated_edge_times)}"
)
print(f"Final on_list count: {len(on_list_time)}")
print(f"Trial count: {len(trials)}")
print(
    "Deleted edge indices relative to the LOW-gap-filled list:"
)
print(automatically_deleted_edge_indices.tolist())
print("Automatic edit summary:")
print(automatic_edit_summary)


In [ ]:
on_list_edge_intervals_seconds = np.diff(on_list_time)
short_interval_indices = np.flatnonzero(
    on_list_edge_intervals_seconds
    < (
        1 - typical_interval_tolerance_fraction
    )
    * typical_edge_interval_seconds
)
long_interval_indices = np.flatnonzero(
    on_list_edge_intervals_seconds
    > (
        1 + typical_interval_tolerance_fraction
    )
    * typical_edge_interval_seconds
)

retained_measured_edge_directions = (
    matched_transition_edge_directions[
        matched_observed_edge_is_retained
    ]
)

print("==============================")
print("First 10 on_list times:")
print(on_list_time[:10])
print("Last 10 on_list times:")
print(on_list_time[-10:])
print(
    "Retained measured rising edges: "
    f"{np.count_nonzero(retained_measured_edge_directions == 1)}"
)
print(
    "Retained measured falling edges: "
    f"{np.count_nonzero(retained_measured_edge_directions == -1)}"
)
print(
    "Final rising edges including interpolation: "
    f"{np.count_nonzero(on_list_edge_directions == 1)}"
)
print(
    "Final falling edges including interpolation: "
    f"{np.count_nonzero(on_list_edge_directions == -1)}"
)
print(f"Remaining short intervals: {len(short_interval_indices)}")
print(f"Remaining long intervals: {len(long_interval_indices)}")
assert len(short_interval_indices) == 0, (
    "Short intervals remain after matching. Inspect them before saving."
)

for long_interval_index in long_interval_indices:
    print(
        f"Long interval at {long_interval_index}: "
        f"{on_list_time[long_interval_index]:.9f} to "
        f"{on_list_time[long_interval_index + 1]:.9f}, "
        f"duration="
        f"{on_list_edge_intervals_seconds[long_interval_index]:.9f} s"
    )

on_list_time_for_export = np.append(
    on_list_time,
    on_list_time[-1] + final_boundary_interval_seconds,
)

final_boundary_direction = -int(on_list_edge_directions[-1])
on_list_edge_directions_for_export = np.append(
    on_list_edge_directions,
    final_boundary_direction,
)

exported_on_list_intervals_seconds = np.diff(
    on_list_time_for_export
)

assert len(on_list_time_for_export) == len(trials) + 1
assert len(on_list_edge_directions_for_export) == (
    len(on_list_time_for_export)
)
assert np.all(
    on_list_edge_directions_for_export[1:]
    == -on_list_edge_directions_for_export[:-1]
)
assert len(exported_on_list_intervals_seconds) == len(trials)
assert np.all(exported_on_list_intervals_seconds > 0)

print(f"Export count: {len(on_list_time_for_export)}")
print(
    "Final synthetic boundary: "
    f"{on_list_time_for_export[-1]:.9f}"
)
print(
    "Final synthetic boundary direction: "
    f"{final_boundary_direction:+d} "
    "(-1 falling, +1 rising)"
)
print(
    "Export boundary directions: "
    f"{np.count_nonzero(on_list_edge_directions_for_export == 1)} "
    "rising, "
    f"{np.count_nonzero(on_list_edge_directions_for_export == -1)} "
    "falling"
)
print(
    "Final export interval: "
    f"{exported_on_list_intervals_seconds[-1] * 1000:.6f} ms"
)


In [ ]:
retained_edge_color = "tab:green"
deleted_edge_color = "tab:red"
inferred_edge_color = "tab:purple"
normal_interval_color = "tab:green"
abnormal_interval_color = "tab:red"

abnormal_interval_mask = (
    np.abs(
        exported_on_list_intervals_seconds
        - typical_edge_interval_seconds
    )
    > typical_interval_tolerance_fraction
    * typical_edge_interval_seconds
)
normal_interval_indices = np.flatnonzero(
    ~abnormal_interval_mask
)
abnormal_interval_indices = np.flatnonzero(
    abnormal_interval_mask
)

figure, interval_axis = plt.subplots(figsize=(12, 4))
figure.patch.set_facecolor("white")
interval_axis.set_facecolor("white")
interval_axis.plot(
    exported_on_list_intervals_seconds,
    color="0.70",
    linewidth=0.8,
    label="Interval sequence",
)
interval_axis.scatter(
    normal_interval_indices,
    exported_on_list_intervals_seconds[normal_interval_indices],
    color=normal_interval_color,
    s=8,
    label="Normal interval",
)
interval_axis.axhline(
    typical_edge_interval_seconds,
    color=normal_interval_color,
    linewidth=1.0,
    linestyle="--",
    label="Typical interval",
)
interval_axis.scatter(
    abnormal_interval_indices,
    exported_on_list_intervals_seconds[abnormal_interval_indices],
    color=abnormal_interval_color,
    marker="x",
    s=24,
    label="Abnormal interval",
)
interval_axis.set_xlabel("Export interval index")
interval_axis.set_ylabel("Interval (seconds)")
figure.tight_layout()
plt.show()


In [ ]:
diagnostic_context_interval_count = 5
diagnostic_context_seconds = (
    diagnostic_context_interval_count
    * typical_edge_interval_seconds
)
diagnostic_events = []

for removed_high_run_start_position in range(
    0,
    len(removed_short_high_edge_sample_indices),
    2,
):
    removed_high_run_start_sample_index = int(
        removed_short_high_edge_sample_indices[
            removed_high_run_start_position
        ]
    )
    removed_high_run_end_sample_index = int(
        removed_short_high_edge_sample_indices[
            removed_high_run_start_position + 1
        ]
    )
    removed_high_run_start_time_seconds = float(
        ADC_continuous_time_stamp_data[
            removed_high_run_start_sample_index
        ]
    )
    removed_high_run_end_time_seconds = float(
        ADC_continuous_time_stamp_data[
            removed_high_run_end_sample_index
        ]
    )
    removed_high_run_duration_ms = (
        removed_high_run_end_time_seconds
        - removed_high_run_start_time_seconds
    ) * 1000
    removed_high_run_number = (
        removed_high_run_start_position // 2 + 1
    )

    diagnostic_events.append(
        {
            "title": (
                f"DELETE short HIGH run {removed_high_run_number}: "
                "2 measured edges, "
                f"{removed_high_run_duration_ms:.3f} ms"
            ),
            "window_start_time_seconds": (
                removed_high_run_start_time_seconds
                - diagnostic_context_seconds
            ),
            "window_end_time_seconds": (
                removed_high_run_end_time_seconds
                + diagnostic_context_seconds
            ),
            "highlight_start_time_seconds": (
                removed_high_run_start_time_seconds
            ),
            "highlight_end_time_seconds": (
                removed_high_run_end_time_seconds
            ),
            "highlight_color": deleted_edge_color,
            "highlight_label": "Removed HIGH span",
        }
    )

for gap_repair_number, gap_repair_information in enumerate(
    applied_gap_repairs,
    start=1,
):
    left_anchor_time_seconds = float(
        ADC_continuous_time_stamp_data[
            gap_repair_information["left_anchor_sample_index"]
        ]
    )
    right_anchor_time_seconds = float(
        ADC_continuous_time_stamp_data[
            gap_repair_information["right_anchor_sample_index"]
        ]
    )
    removed_measured_interior_edge_count = (
        gap_repair_information["observed_interval_count"] - 1
    )
    inferred_interior_edge_count = (
        gap_repair_information["expected_interval_count"] - 1
    )

    diagnostic_events.append(
        {
            "title": (
                f"REPLACE gap {gap_repair_number}: "
                f"{gap_repair_information['observed_interval_count']} "
                "-> "
                f"{gap_repair_information['expected_interval_count']} intervals; "
                f"delete {removed_measured_interior_edge_count} measured, "
                f"infer {inferred_interior_edge_count}, net "
                f"{gap_repair_information['net_transition_count_change']:+d}"
            ),
            "window_start_time_seconds": (
                left_anchor_time_seconds
                - diagnostic_context_seconds
            ),
            "window_end_time_seconds": (
                right_anchor_time_seconds
                + diagnostic_context_seconds
            ),
            "highlight_start_time_seconds": (
                left_anchor_time_seconds
            ),
            "highlight_end_time_seconds": (
                right_anchor_time_seconds
            ),
            "highlight_color": inferred_edge_color,
            "highlight_label": "Interpolated anchor span",
        }
    )

for long_interval_number, long_interval_index in enumerate(
    long_interval_indices,
    start=1,
):
    long_interval_start_time_seconds = float(
        on_list_time[long_interval_index]
    )
    long_interval_end_time_seconds = float(
        on_list_time[long_interval_index + 1]
    )
    long_interval_duration_seconds = (
        long_interval_end_time_seconds
        - long_interval_start_time_seconds
    )

    diagnostic_events.append(
        {
            "title": (
                f"PRESERVE long gap {long_interval_number}: "
                f"{long_interval_duration_seconds:.6f} s; "
                "no interpolation"
            ),
            "window_start_time_seconds": (
                long_interval_start_time_seconds
                - diagnostic_context_seconds
            ),
            "window_end_time_seconds": (
                long_interval_end_time_seconds
                + diagnostic_context_seconds
            ),
            "highlight_start_time_seconds": (
                long_interval_start_time_seconds
            ),
            "highlight_end_time_seconds": (
                long_interval_end_time_seconds
            ),
            "highlight_color": abnormal_interval_color,
            "highlight_label": "Preserved gap",
        }
    )

boundary_comparison_context_seconds = 1.0
if trim_start_count > 0:
    old_first_transition_time_seconds = float(transition_times[0])
    revised_first_transition_time_seconds = float(on_list_time[0])
    diagnostic_events.append(
        {
            "title": (
                f"TRIM START: remove {trim_start_count} measured "
                "edges; old first -> revised first"
            ),
            "window_start_time_seconds": (
                old_first_transition_time_seconds
                - boundary_comparison_context_seconds
            ),
            "window_end_time_seconds": (
                revised_first_transition_time_seconds
                + boundary_comparison_context_seconds
            ),
            "interval_indices": np.arange(
                periodic_support_interval_count
            ),
            "interval_title": (
                "First post-trim intervals"
            ),
            "highlight_start_time_seconds": (
                old_first_transition_time_seconds
            ),
            "highlight_end_time_seconds": (
                revised_first_transition_time_seconds
            ),
            "highlight_color": deleted_edge_color,
            "highlight_label": "Removed start span",
        }
    )

revised_last_boundary_time_seconds = float(
    on_list_time_for_export[-1]
)
last_retained_edge_direction_label = (
    "rising"
    if on_list_edge_directions[-1] == 1
    else "falling"
)
final_boundary_direction_label = (
    "rising"
    if final_boundary_direction == 1
    else "falling"
)
last_export_interval_indices = np.arange(
    len(exported_on_list_intervals_seconds)
    - periodic_support_interval_count,
    len(exported_on_list_intervals_seconds),
)

if trim_end_count > 0:
    old_last_transition_time_seconds = float(transition_times[-1])
    diagnostic_events.append(
        {
            "title": (
                f"TRIM END OVERVIEW: remove {trim_end_count} measured "
                "edges; revised final boundary -> old last"
            ),
            "window_start_time_seconds": (
                revised_last_boundary_time_seconds
                - boundary_comparison_context_seconds
            ),
            "window_end_time_seconds": (
                old_last_transition_time_seconds
                + boundary_comparison_context_seconds
            ),
            "interval_indices": last_export_interval_indices,
            "interval_title": (
                "Last post-trim intervals including final boundary"
            ),
            "additional_inferred_edge_times": np.array(
                [revised_last_boundary_time_seconds]
            ),
            "highlight_start_time_seconds": (
                revised_last_boundary_time_seconds
            ),
            "highlight_end_time_seconds": (
                old_last_transition_time_seconds
            ),
            "highlight_color": deleted_edge_color,
            "highlight_label": "Removed end span",
        }
    )

diagnostic_events.append(
    {
        "title": (
            "FINAL BOUNDARY DETAIL: measured "
            f"{last_retained_edge_direction_label} -> synthetic "
            f"{final_boundary_direction_label}; "
            f"{exported_on_list_intervals_seconds[-1] * 1000:.3f} ms"
        ),
        "window_start_time_seconds": (
            revised_last_boundary_time_seconds
            - boundary_comparison_context_seconds
        ),
        "window_end_time_seconds": (
            revised_last_boundary_time_seconds
            + boundary_comparison_context_seconds
        ),
        "interval_indices": last_export_interval_indices,
        "interval_title": (
            "Last post-trim intervals including final boundary"
        ),
        "additional_inferred_edge_times": np.array(
            [revised_last_boundary_time_seconds]
        ),
        "highlight_start_time_seconds": (
            revised_last_boundary_time_seconds
        ),
        "highlight_end_time_seconds": (
            revised_last_boundary_time_seconds
            + boundary_comparison_context_seconds
        ),
        "highlight_color": deleted_edge_color,
        "highlight_label": "Outside exported trial block",
        "show_export_digital_state": True,
    }
)

print(
    "Local plots cover HIGH-run removal, boundary trim, "
    "gap repair, and preserved long gaps."
)
print(
    "Local event context: +/-"
    f"{diagnostic_context_seconds:.6f} seconds "
    f"({diagnostic_context_interval_count} typical intervals)."
)
print(
    "Boundary comparison context: +/-"
    f"{boundary_comparison_context_seconds:.3f} seconds."
)
print(
    "End diagnostics include a full trim overview and a "
    "fixed-scale final-boundary detail."
)
print(
    "LOW-gap fill collapsed "
    f"{raw_transition_count - len(filled_transition_sample_indices)} "
    "raw crack transitions; they are summarized, not plotted "
    "one by one."
)
print(f"Diagnostic event count: {len(diagnostic_events)}")
for diagnostic_event_number, diagnostic_event in enumerate(
    diagnostic_events,
    start=1,
):
    print(
        f"{diagnostic_event_number}: "
        f"{diagnostic_event['title']}"
    )

    diagnostic_start_time_seconds = diagnostic_event[
        "window_start_time_seconds"
    ]
    diagnostic_end_time_seconds = diagnostic_event[
        "window_end_time_seconds"
    ]
    diagnostic_start_sample_index = np.searchsorted(
        ADC_continuous_time_stamp_data,
        diagnostic_start_time_seconds,
        side="left",
    )
    diagnostic_end_sample_index = np.searchsorted(
        ADC_continuous_time_stamp_data,
        diagnostic_end_time_seconds,
        side="right",
    )

    diagnostic_time = ADC_continuous_time_stamp_data[
        diagnostic_start_sample_index:diagnostic_end_sample_index
    ]
    diagnostic_photodiode_data = photodiode_data[
        diagnostic_start_sample_index:diagnostic_end_sample_index
    ]
    diagnostic_clean_digital_data = photodiode_is_high_clean[
        diagnostic_start_sample_index:diagnostic_end_sample_index
    ]

    visible_observed_edge_mask = (
        (
            final_observed_transition_sample_indices
            >= diagnostic_start_sample_index
        )
        & (
            final_observed_transition_sample_indices
            < diagnostic_end_sample_index
        )
    )
    visible_observed_edge_sample_indices = (
        final_observed_transition_sample_indices[
            visible_observed_edge_mask
        ]
    )
    visible_observed_edge_times = (
        ADC_continuous_time_stamp_data[
            visible_observed_edge_sample_indices
        ]
    )
    visible_observed_edge_directions = (
        clean_digital_state_changes[
            visible_observed_edge_sample_indices - 1
        ]
    )
    visible_rising_edge_mask = (
        visible_observed_edge_directions == 1
    )
    visible_falling_edge_mask = (
        visible_observed_edge_directions == -1
    )

    visible_deleted_edge_mask = (
        (
            automatically_deleted_edge_sample_indices
            >= diagnostic_start_sample_index
        )
        & (
            automatically_deleted_edge_sample_indices
            < diagnostic_end_sample_index
        )
    )
    visible_deleted_edge_sample_indices = (
        automatically_deleted_edge_sample_indices[
            visible_deleted_edge_mask
        ]
    )
    visible_deleted_edge_times = (
        ADC_continuous_time_stamp_data[
            visible_deleted_edge_sample_indices
        ]
    )
    visible_deleted_edge_directions = (
        automatically_deleted_edge_directions[
            visible_deleted_edge_mask
        ]
    )
    visible_deleted_rising_edge_mask = (
        visible_deleted_edge_directions == 1
    )
    visible_deleted_falling_edge_mask = (
        visible_deleted_edge_directions == -1
    )

    diagnostic_inferred_edge_times = interpolated_edge_times
    if "additional_inferred_edge_times" in diagnostic_event:
        diagnostic_inferred_edge_times = np.concatenate(
            (
                diagnostic_inferred_edge_times,
                diagnostic_event["additional_inferred_edge_times"],
            )
        )
    visible_inferred_edge_times = diagnostic_inferred_edge_times[
        (
            diagnostic_inferred_edge_times
            >= diagnostic_start_time_seconds
        )
        & (
            diagnostic_inferred_edge_times
            <= diagnostic_end_time_seconds
        )
    ]

    show_export_digital_state = diagnostic_event.get(
        "show_export_digital_state",
        False,
    )
    if show_export_digital_state:
        first_visible_export_boundary_index = int(
            np.searchsorted(
                on_list_time_for_export,
                diagnostic_start_time_seconds,
                side="left",
            )
        )
        export_boundary_stop_index = int(
            np.searchsorted(
                on_list_time_for_export,
                diagnostic_end_time_seconds,
                side="right",
            )
        )
        visible_export_boundary_times = on_list_time_for_export[
            first_visible_export_boundary_index:
            export_boundary_stop_index
        ]
        visible_export_boundary_directions = (
            on_list_edge_directions_for_export[
                first_visible_export_boundary_index:
                export_boundary_stop_index
            ]
        )

        if first_visible_export_boundary_index == 0:
            export_state_at_window_start = int(
                on_list_edge_directions_for_export[0] == -1
            )
        else:
            export_state_at_window_start = int(
                on_list_edge_directions_for_export[
                    first_visible_export_boundary_index - 1
                ]
                == 1
            )

        export_states_after_visible_boundaries = (
            visible_export_boundary_directions == 1
        ).astype(np.int8)
        export_state_at_window_end = (
            int(export_states_after_visible_boundaries[-1])
            if len(export_states_after_visible_boundaries) > 0
            else export_state_at_window_start
        )
        export_digital_plot_times = np.concatenate(
            (
                [diagnostic_start_time_seconds],
                visible_export_boundary_times,
                [diagnostic_end_time_seconds],
            )
        )
        export_digital_plot_states = np.concatenate(
            (
                [export_state_at_window_start],
                export_states_after_visible_boundaries,
                [export_state_at_window_end],
            )
        )

    if "interval_indices" in diagnostic_event:
        diagnostic_interval_indices = diagnostic_event[
            "interval_indices"
        ]
        diagnostic_interval_title = diagnostic_event[
            "interval_title"
        ]
    else:
        diagnostic_interval_indices = np.flatnonzero(
            (
                on_list_time_for_export[:-1]
                >= diagnostic_start_time_seconds
            )
            & (
                on_list_time_for_export[1:]
                <= diagnostic_end_time_seconds
            )
        )
        diagnostic_interval_title = "Final intervals in window"

    diagnostic_interval_seconds = exported_on_list_intervals_seconds[
        diagnostic_interval_indices
    ]
    diagnostic_interval_milliseconds = (
        diagnostic_interval_seconds * 1000
    )
    diagnostic_interval_is_abnormal = (
        np.abs(
            diagnostic_interval_seconds
            - typical_edge_interval_seconds
        )
        > typical_interval_tolerance_fraction
        * typical_edge_interval_seconds
    )

    figure = plt.figure(figsize=(16, 6))
    diagnostic_grid = figure.add_gridspec(
        2,
        2,
        width_ratios=[4, 1.35],
        height_ratios=[3, 1],
    )
    ADC_axis = figure.add_subplot(diagnostic_grid[0, 0])
    digital_axis = figure.add_subplot(
        diagnostic_grid[1, 0],
        sharex=ADC_axis,
    )
    interval_axis = figure.add_subplot(diagnostic_grid[:, 1])
    figure.patch.set_facecolor("white")
    for diagnostic_axis in (
        ADC_axis,
        digital_axis,
        interval_axis,
    ):
        diagnostic_axis.set_facecolor("white")
    figure.suptitle(
        f"{diagnostic_event_number}/{len(diagnostic_events)}: "
        f"{diagnostic_event['title']}"
    )

    for diagnostic_highlight_time_seconds in (
        diagnostic_event["highlight_start_time_seconds"],
        diagnostic_event["highlight_end_time_seconds"],
    ):
        ADC_axis.axvline(
            diagnostic_highlight_time_seconds,
            color=diagnostic_event["highlight_color"],
            linewidth=0.8,
            linestyle="--",
            alpha=0.45,
        )
        digital_axis.axvline(
            diagnostic_highlight_time_seconds,
            color=diagnostic_event["highlight_color"],
            linewidth=0.8,
            linestyle="--",
            alpha=0.45,
        )

    ADC_axis.plot(
        diagnostic_time,
        diagnostic_photodiode_data,
        color="C0",
        linewidth=0.9,
        label="Raw ADC",
    )
    ADC_axis.axhline(
        edge_threshold,
        color="0.35",
        linewidth=0.9,
        linestyle="--",
        label="Edge threshold",
    )
    ADC_axis.scatter(
        visible_observed_edge_times[visible_rising_edge_mask],
        photodiode_data[
            visible_observed_edge_sample_indices[
                visible_rising_edge_mask
            ]
        ],
        color=retained_edge_color,
        marker="^",
        s=55,
        zorder=5,
        label="Retained rising edge",
    )
    ADC_axis.scatter(
        visible_observed_edge_times[visible_falling_edge_mask],
        photodiode_data[
            visible_observed_edge_sample_indices[
                visible_falling_edge_mask
            ]
        ],
        color=retained_edge_color,
        marker="v",
        s=55,
        zorder=5,
        label="Retained falling edge",
    )
    ADC_axis.scatter(
        visible_deleted_edge_times[
            visible_deleted_rising_edge_mask
        ],
        photodiode_data[
            visible_deleted_edge_sample_indices[
                visible_deleted_rising_edge_mask
            ]
        ],
        facecolors="none",
        edgecolors=deleted_edge_color,
        linewidths=1.6,
        marker="^",
        s=70,
        zorder=6,
        label="Deleted rising edge",
    )
    ADC_axis.scatter(
        visible_deleted_edge_times[
            visible_deleted_falling_edge_mask
        ],
        photodiode_data[
            visible_deleted_edge_sample_indices[
                visible_deleted_falling_edge_mask
            ]
        ],
        facecolors="none",
        edgecolors=deleted_edge_color,
        linewidths=1.6,
        marker="v",
        s=70,
        zorder=6,
        label="Deleted falling edge",
    )

    for inferred_edge_number, inferred_edge_time in (
        enumerate(visible_inferred_edge_times)
    ):
        ADC_axis.axvline(
            inferred_edge_time,
            color=inferred_edge_color,
            linewidth=1.0,
            linestyle=":",
            alpha=0.9,
            label=(
                "Inferred edge or boundary (not measured)"
                if inferred_edge_number == 0
                else None
            ),
        )

    ADC_axis.scatter(
        visible_inferred_edge_times,
        np.full(len(visible_inferred_edge_times), edge_threshold),
        facecolors="white",
        edgecolors=inferred_edge_color,
        linewidths=1.4,
        marker="D",
        s=42,
        zorder=7,
    )

    digital_axis.step(
        diagnostic_time,
        diagnostic_clean_digital_data.astype(np.int8),
        where="post",
        color="0.15",
        linewidth=1.2,
        label="Measured clean DI",
    )
    if show_export_digital_state:
        digital_axis.step(
            export_digital_plot_times,
            export_digital_plot_states,
            where="post",
            color=inferred_edge_color,
            linewidth=1.5,
            linestyle="--",
            alpha=0.9,
            label="Logical export DI",
        )
    for observed_edge_time in visible_observed_edge_times:
        digital_axis.axvline(
            observed_edge_time,
            color=retained_edge_color,
            linewidth=0.8,
            alpha=0.35,
        )
    for deleted_edge_time in visible_deleted_edge_times:
        digital_axis.axvline(
            deleted_edge_time,
            color=deleted_edge_color,
            linewidth=0.9,
            linestyle="--",
            alpha=0.65,
        )
    for inferred_edge_time in visible_inferred_edge_times:
        digital_axis.axvline(
            inferred_edge_time,
            color=inferred_edge_color,
            linewidth=0.9,
            linestyle=":",
            alpha=0.75,
        )

    typical_edge_interval_milliseconds = (
        typical_edge_interval_seconds * 1000
    )
    interval_axis.axhline(
        typical_edge_interval_milliseconds,
        color=normal_interval_color,
        linewidth=1.0,
        linestyle="--",
        label="Typical interval",
    )
    interval_axis.plot(
        diagnostic_interval_indices,
        diagnostic_interval_milliseconds,
        color="0.55",
        linewidth=0.8,
    )
    interval_axis.scatter(
        diagnostic_interval_indices[
            ~diagnostic_interval_is_abnormal
        ],
        diagnostic_interval_milliseconds[
            ~diagnostic_interval_is_abnormal
        ],
        color=normal_interval_color,
        s=24,
        label="Normal",
    )
    interval_axis.scatter(
        diagnostic_interval_indices[
            diagnostic_interval_is_abnormal
        ],
        diagnostic_interval_milliseconds[
            diagnostic_interval_is_abnormal
        ],
        color=abnormal_interval_color,
        marker="x",
        s=45,
        label="Abnormal",
    )
    interval_axis.set_title(diagnostic_interval_title)
    interval_axis.set_xlabel("Final interval index")
    interval_axis.set_ylabel("Interval (ms)")
    interval_axis.grid(alpha=0.2)

    ADC_axis.set_ylabel("ADC value")
    ADC_axis.grid(axis="x", alpha=0.2)
    digital_axis.set_ylim(-0.15, 1.15)
    digital_axis.set_yticks([0, 1], labels=["LOW", "HIGH"])
    if show_export_digital_state:
        digital_axis.set_ylabel("Measured / logical DI")
    else:
        digital_axis.set_ylabel("Clean DI")
    digital_axis.set_xlabel("ADC timestamp (seconds)")
    digital_axis.ticklabel_format(
        axis="x",
        style="plain",
        useOffset=False,
    )
    digital_axis.set_xlim(
        diagnostic_start_time_seconds,
        diagnostic_end_time_seconds,
    )
    digital_axis.grid(axis="x", alpha=0.2)

    figure.tight_layout()
    plt.show()


In [ ]:
on_list_time_output_file = f"{base_dir}/data/on_list_times.npy"

if is_save_on_list_time:
    np.save(on_list_time_output_file, on_list_time_for_export)
    print(
        f"Saved {len(on_list_time_for_export)} boundaries to "
        f"{on_list_time_output_file}"
    )
else:
    print("Preview complete; on_list_times.npy was not changed.")
    print(
        "Set is_save_on_list_time = True and run all cells "
        "to save the verified result."
    )
